<a href="https://colab.research.google.com/github/Loopinlogix/Market_Analysis_Project-2/blob/main/Stock_Market_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Stock Market Analysis Project 2


## Intro to this Project

This notebook is all about digging into some historical stock market data. We're gonna do a bunch of things: grab the data, clean it up (get rid of weird errors and bad values), find any crazy outliers, check for duplicates, cook up some new features from the existing data, make sure everything's on the same scale, and then split it all up so we can eventually build some machine learning models.

Basically, we've got two main data files: one with general info about stocks (`historical_stocks.csv`) like where they're traded, their names, what industry they're in, etc., and another with the daily prices and trading volumes (`historical_stock_prices.csv`).

The whole point here is to take all that raw, messy stock info and turn it into something neat and organized, packed with useful features. This way, we'll have a solid dataset ready to go for training models to try and figure out what the stock market might do next.

In [ ]:

#Github

#Github
!apt-get install -y git
!git config --global user.email "crystal_macneil@hotmail.com"
!git config --global user.name "Crystal MacNeil"

!git clone https://github.com/Loopinlogix/Market_Analysis_Project-2.git
%cd Market_Analysis_Project-2
!ls


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Cloning into 'Market_Analysis_Project-2'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/Market_Analysis_Project-2
README.md


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("=" * 75)
print("STEP 1: LOAD AND MERGE THE DATA")
print("=" * 75)

# Load two datasets
stocks = pd.read_csv('historical_stocks.csv')
prices = pd.read_csv('historical_stock_prices.csv')

print(prices.head())
print(stocks.head())
print(prices.info())
print(stocks.info())

# Clean up column names (remove extra spaces, make lowercase)
stocks.columns = stocks.columns.str.strip().str.lower()
prices.columns = prices.columns.str.strip().str.lower()

print(f"Stocks info shape: {stocks.shape}")
print(f"Prices info shape: {prices.shape}")


#remove "fake" header rows.
initial_rows = len(stocks)
stocks = stocks[stocks['ticker'].str.upper() != 'SYMBOL']
stocks = stocks[stocks['ticker'].str.upper() != 'TICKER']
stocks['sector'] = stocks['sector'].replace('N/A', np.nan)
stocks['industry'] = stocks['industry'].replace('N/A', np.nan)
print(f"Removed {initial_rows - len(stocks)} repeated header rows")

# merge the two datasets using the 'ticker' column
df = pd.merge(prices, stocks, on='ticker', how='left')
print(f"Combined dataset shape: {df.shape}")


print("=" * 60)
print("ADVANCED STEP 2: MISSING VALUE IMPUTATION")
print("=" * 60)

prices_adv = prices.copy()
# Convert 'date' column to datetime and set as index for time-weighted interpolation
prices_adv['date'] = pd.to_datetime(prices_adv['date'], errors='coerce') # Coerce invalid dates to NaT
prices_adv = prices_adv.dropna(subset=['date']) # Drop rows where date conversion resulted in NaT
prices_adv = prices_adv.set_index('date').sort_index()

def advanced_impute(df):
    numeric = ['open','high','low','close','volume']

    # Time-based interpolation
    df[numeric] = df[numeric].interpolate(method='time')

    # Forward + backward fill
    df[numeric] = df[numeric].ffill().bfill()

    return df

prices_adv = prices_adv.groupby('ticker', group_keys=False).apply(advanced_impute)

print("Remaining missing values:\n", prices_adv.isnull().sum())

print("=" * 60)
print("ADVANCED STEP 3: OUTLIER DETECTION & CAPPING")
print("=" * 60)

def cap_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return series.clip(lower, upper)

for col in ['close', 'volume']:
    prices_adv[col] = prices_adv.groupby('ticker')[col].transform(cap_iqr)

print("Outliers capped for close & volume.")

print("=" * 60)
print("ADVANCED STEP 4: ERROR CHECKING & CORRECTION")
print("=" * 60)

def fix_errors(df):
    numeric = ['open','high','low','close','volume']

    mask = (
        (df['open'] <= 0) |
        (df['high'] <= 0) |
        (df['low'] <= 0) |
        (df['close'] <= 0) |
        (df['volume'] < 0) |
        (df['high'] < df['low'])
    )

    df.loc[mask, numeric] = np.nan

    df[numeric] = df[numeric].interpolate(method='time').ffill().bfill()

    return df

prices_adv = prices_adv.groupby('ticker', group_keys=False).apply(fix_errors)

print("Errors corrected.")



print("=" * 60)
print("ADVANCED STEP 5: FEATURE ENGINEERING")
print("=" * 60)

prices_fe = prices_adv.copy()

# Daily returns
prices_fe['return'] = prices_fe.groupby('ticker')['close'].pct_change()

# Rolling mean & volatility
prices_fe['ma_20'] = prices_fe.groupby('ticker')['close'].transform(lambda s: s.rolling(20).mean())
prices_fe['vol_20'] = prices_fe.groupby('ticker')['return'].transform(lambda s: s.rolling(20).std())

# RSI (14-day)
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))

prices_fe['rsi_14'] = prices_fe.groupby('ticker')['close'].transform(compute_rsi)

print("Features added: return, ma_20, vol_20, rsi_14")


print("=" * 60)
print("ADVANCED STEP 6: NORMALIZATION")
print("=" * 60)

from sklearn.preprocessing import StandardScaler

numeric_cols = [
    'open','high','low','close','volume',
    'return','ma_20','vol_20','rsi_14'
]

prices_model = prices_fe.dropna(subset=numeric_cols).copy()

scaler = StandardScaler()
prices_model[numeric_cols] = scaler.fit_transform(prices_model[numeric_cols])

print("Numeric fields standardized.")


print("=" * 60)
print("ADVANCED STEP 7: ENCODING CATEGORICAL VARIABLES")
print("=" * 60)

cat_cols = [col for col in ['sector','industry'] if col in stocks.columns]

stocks_enc = pd.get_dummies(stocks, columns=cat_cols, drop_first=True)

prices_model_reset = prices_model.reset_index()

merged_final = pd.merge(prices_model_reset, stocks_enc, on='ticker', how='left')

print("Merged dataset shape:", merged_final.shape)


print("=" * 60)
print("ADVANCED STEP 8: DATA SPLITTING")
print("=" * 60)

merged_final = merged_final.sort_values('date')

merged_final['target_next_return'] = merged_final.groupby('ticker')['return'].shift(-1)
merged_final = merged_final.dropna(subset=['target_next_return'])

feature_cols = numeric_cols + [col for col in merged_final.columns if col.startswith('sector_') or col.startswith('industry_')]

X = merged_final[feature_cols]
y = merged_final['target_next_return']

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, shuffle=False)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)



print("=" * 60)
print("ADVANCED STEP 9: SAVE CLEAN DATA")
print("=" * 60)

merged_final.to_csv('stocks_clean_full.csv', index=False)
X_train.to_csv('X_train.csv', index=False)
X_val.to_csv('X_val.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_val.to_csv('y_val.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("All cleaned datasets saved.")


print("=" * 60)
print("PLOT: Missing Values Before vs After Imputation")
print("=" * 60)

missing_before = prices[['open','high','low','close','volume']].isnull().sum()
missing_after = prices_adv[['open','high','low','close','volume']].isnull().sum()

plt.figure(figsize=(10,5))
plt.bar(missing_before.index, missing_before.values, alpha=0.6, label='Before')
plt.bar(missing_after.index, missing_after.values, alpha=0.6, label='After')
plt.title("Missing Values Before vs After Imputation")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

print("=" * 60)
print("PLOT: Outlier Capping Effect on Close Prices")
print("=" * 60)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.boxplot(data=prices, y='close')
plt.title("Before Outlier Capping")

plt.subplot(1,2,2)
sns.boxplot(data=prices_adv, y='close')
plt.title("After Outlier Capping")

plt.tight_layout()
plt.show()

print("=" * 60)
print("PLOT: Volume Distribution Before vs After Outlier Capping")
print("=" * 60)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.hist(prices['volume'], bins=50, color='salmon', alpha=0.7)
plt.title("Volume Before Capping")

plt.subplot(1,2,2)
plt.hist(prices_adv['volume'], bins=50, color='steelblue', alpha=0.7)
plt.title("Volume After Capping")

plt.tight_layout()
plt.show()

print("=" * 60)
print("PLOT: Error Correction Counts")
print("=" * 60)

errors_before = (
    (prices['high'] < prices['low']) |
    (prices['open'] <= 0) |
    (prices['close'] <= 0)
).sum()

errors_after = (
    (prices_adv['high'] < prices_adv['low']) |
    (prices_adv['open'] <= 0) |
    (prices_adv['close'] <= 0)
).sum()

plt.figure(figsize=(6,5))
plt.bar(['Before', 'After'], [errors_before, errors_after], color=['red','green'])
plt.title("Error Counts Before vs After Correction")
plt.ylabel("Number of Errors")
plt.tight_layout()
plt.show()

print("=" * 60)
print("PLOT: Rolling Mean & Volatility")
print("=" * 60)

ticker_sample = prices_fe[prices_fe['ticker'] == prices_fe['ticker'].iloc[0]]

plt.figure(figsize=(14,6))
plt.plot(ticker_sample.index, ticker_sample['close'], label='Close Price', alpha=0.6)
plt.plot(ticker_sample.index, ticker_sample['ma_20'], label='20-Day MA', linewidth=2)
plt.title("Close Price vs 20-Day Moving Average")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14,6))
plt.plot(ticker_sample.index, ticker_sample['vol_20'], color='purple')
plt.title("20-Day Volatility")
plt.tight_layout()
plt.show()

print("=" * 60)
print("PLOT: RSI Indicator")
print("=" * 60)

plt.figure(figsize=(14,5))
plt.plot(ticker_sample.index, ticker_sample['rsi_14'], color='orange')
plt.axhline(70, color='red', linestyle='--', alpha=0.5)
plt.axhline(30, color='green', linestyle='--', alpha=0.5)
plt.title("RSI (14-Day)")
plt.tight_layout()
plt.show()

print("=" * 60)
print("PLOT: Normalization Effect")
print("=" * 60)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.boxplot(data=prices_fe[['open','high','low','close','volume']])
plt.title("Before Scaling")

plt.subplot(1,2,2)
sns.boxplot(data=prices_model[['open','high','low','close','volume']])
plt.title("After Scaling")

plt.tight_layout()
plt.show()

print("=" * 60)
print("PLOT: Final Correlation Heatmap")
print("=" * 60)

plt.figure(figsize=(10,8))
sns.heatmap(
    merged_final[numeric_cols].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)
plt.title("Correlation Matrix After Full Cleaning")
plt.tight_layout()
plt.show()










STEP 1: LOAD AND MERGE THE DATA
  ticker   open  close  adj_close    low   high     volume        date
0    AHH  11.50  11.58   8.493155  11.25  11.68  4633900.0  2013-05-08
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800.0  2013-05-09
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100.0  2013-05-10
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400.0  2013-05-13
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100.0  2013-05-14
  ticker exchange                                    name             sector  \
0    PIH   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
1  PIHPP   NASDAQ  1347 PROPERTY INSURANCE HOLDINGS, INC.            FINANCE   
2   TURN   NASDAQ                180 DEGREE CAPITAL CORP.            FINANCE   
3   FLWS   NASDAQ                 1-800 FLOWERS.COM, INC.  CONSUMER SERVICES   
4   FCCY   NASDAQ           1ST CONSTITUTION BANCORP (NJ)            FINANCE   

                     industry  
0  PROPERTY-CASUALTY INSURERS